# A Data-Driven Cyber Risk Analytics and Decision Support Framework
## for Digital Banking Environments
**Gerta Rustemi | Student ID: 070124011 | MSc Computer Science, UNYT**

---

# PHASE 2 — Model Training, Evaluation & Risk Framework

**Prerequisites:** Phase 1 notebook must be completed first.  
All inputs are loaded from `My Drive/Thesis/preprocessed/`

This notebook covers:
1. Environment setup and loading Phase 1 outputs
2. **Model 1 — Random Forest** (bagging ensemble)
3. **Model 2 — XGBoost** (gradient boosting)
4. **Model 3 — LSTM** (recurrent neural network for sequential patterns)
5. **Model 4 — Isolation Forest** (unsupervised anomaly detection)
6. Comparative evaluation across all models
7. SHAP explainability analysis (Random Forest + XGBoost)
8. Risk Tier Decision-Support Framework (Critical / High / Medium / Low)
9. Final summary and dissertation outputs

**Dataset:** CICIDS2017 | **Training set:** 5,836,260 records (post-SMOTE) | **Test set:** 195,180 records  
**Classes:** 10 (BENIGN + 9 attack categories)

---
## CELL 1 — Install Libraries
> Run this first and restart runtime if prompted.

In [ ]:
!pip install xgboost shap imbalanced-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import os
import time
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import label_binarize

# XGBoost
import xgboost as xgb

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# SHAP
import shap

# Style
plt.style.use('seaborn-v0_8-whitegrid')
TEAL      = '#0ABAB5'
DARK_TEAL = '#088A87'
N_CLASSES = 10

print(f'✅ All libraries imported')
print(f'TensorFlow: {tf.__version__} | XGBoost: {xgb.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## CELL 2 — Mount Drive and Load Phase 1 Outputs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- UPDATE if your path differs ----
PREP_DIR  = '/content/drive/MyDrive/Thesis/preprocessed/'
MODEL_DIR = '/content/drive/MyDrive/Thesis/models/'
FIG_DIR   = '/content/drive/MyDrive/Thesis/figures/'
# -------------------------------------

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR,   exist_ok=True)

print('Loading Phase 1 outputs...')

# Features and labels
X_train_sm  = np.load(f'{PREP_DIR}X_train_sm.npy')   # (5836260, 78) SMOTE-balanced
y_train_sm  = np.load(f'{PREP_DIR}y_train_sm.npy')   # (5836260,)
X_test      = np.load(f'{PREP_DIR}X_test.npy')        # (195180, 78)
y_test      = np.load(f'{PREP_DIR}y_test.npy')        # (195180,)
X_train_pca = np.load(f'{PREP_DIR}X_train_pca.npy')  # (5836260, 16) PCA-reduced
X_test_pca  = np.load(f'{PREP_DIR}X_test_pca.npy')   # (195180, 16)

# Fitted objects
le               = joblib.load(f'{PREP_DIR}label_encoder.pkl')
scaler           = joblib.load(f'{PREP_DIR}scaler.pkl')
pca              = joblib.load(f'{PREP_DIR}pca.pkl')
selected_features = joblib.load(f'{PREP_DIR}selected_features.pkl')
feature_names    = pd.read_csv(f'{PREP_DIR}feature_names.csv').iloc[:, 0].tolist()

CLASS_NAMES = list(le.classes_)  # 10 class names

print(f'✅ All Phase 1 outputs loaded')
print(f'Training set (SMOTE): {X_train_sm.shape[0]:,} rows × {X_train_sm.shape[1]} features')
print(f'Test set:             {X_test.shape[0]:,} rows × {X_test.shape[1]} features')
print(f'PCA training set:     {X_train_pca.shape[0]:,} rows × {X_train_pca.shape[1]} components')
print(f'Classes: {CLASS_NAMES}')

---
## CELL 3 — Helper Functions
Shared evaluation and plotting utilities used by all four models.

In [ ]:
# ── Evaluation helper ──────────────────────────────────────────────────────
def evaluate_model(name, y_true, y_pred, y_prob=None, training_time=None):
    """Compute and print all dissertation evaluation metrics."""
    results = {
        'model': name,
        'macro_f1':    f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'macro_precision': precision_score(y_true, y_pred, average='macro',    zero_division=0),
        'macro_recall':    recall_score(y_true, y_pred, average='macro',       zero_division=0),
        'fpr': None,
        'auc_roc': None,
        'training_time_s': training_time
    }

    # False Positive Rate (macro average)
    cm = confusion_matrix(y_true, y_pred)
    fpr_per_class = []
    for i in range(len(cm)):
        fp = cm[:, i].sum() - cm[i, i]
        tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fpr_per_class.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    results['fpr'] = np.mean(fpr_per_class)

    # AUC-ROC (requires probability scores)
    if y_prob is not None:
        try:
            results['auc_roc'] = roc_auc_score(
                label_binarize(y_true, classes=list(range(N_CLASSES))),
                y_prob, average='macro', multi_class='ovr'
            )
        except Exception:
            results['auc_roc'] = None

    print(f'\n{"="*55}')
    print(f'  {name} — Evaluation Results')
    print(f'{"="*55}')
    print(f'  Macro F1-score:      {results["macro_f1"]:.4f}')
    print(f'  Weighted F1-score:   {results["weighted_f1"]:.4f}')
    print(f'  Macro Precision:     {results["macro_precision"]:.4f}')
    print(f'  Macro Recall:        {results["macro_recall"]:.4f}')
    print(f'  Mean FPR:            {results["fpr"]:.4f}')
    if results['auc_roc']:
        print(f'  AUC-ROC (macro):     {results["auc_roc"]:.4f}')
    if training_time:
        print(f'  Training time:       {training_time:.1f}s')
    print(f'\n  Per-class report:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    return results


# ── Confusion matrix plot ──────────────────────────────────────────────────
def plot_confusion_matrix(name, y_true, y_pred, filename):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ['Counts', 'Normalised (row %)'],
        ['d', '.2f']
    ):
        sns.heatmap(
            data, annot=True, fmt=fmt, cmap='YlOrRd',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, ax=ax, cbar=True
        )
        ax.set_title(f'{name} — Confusion Matrix ({title})', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=11)
        ax.set_xlabel('Predicted Label', fontsize=11)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}{filename}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {filename}')


# ── Precision-Recall curve ─────────────────────────────────────────────────
def plot_pr_curves(name, y_true, y_prob, filename):
    y_bin = label_binarize(y_true, classes=list(range(N_CLASSES)))
    fig, ax = plt.subplots(figsize=(10, 7))
    cmap = plt.cm.get_cmap('tab10', N_CLASSES)
    for i in range(N_CLASSES):
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        ax.plot(rec, prec, color=cmap(i), lw=1.5,
                label=f'{CLASS_NAMES[i]} (AP={ap:.2f})')
    ax.set_xlabel('Recall', fontsize=12)
    ax.set_ylabel('Precision', fontsize=12)
    ax.set_title(f'{name} — Precision-Recall Curves (per class)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}{filename}', dpi=150, bbox_inches='tight')
    plt.show()


# Storage for comparative table
all_results = []
print('✅ Helper functions defined')

In [ ]:
# RECOVERY — Load pre-trained Random Forest from Drive (skip retraining)
import joblib
print('Loading saved Random Forest model...')
rf = joblib.load(f'{MODEL_DIR}random_forest.pkl')
rf_time = 0  # Training time unknown after reload
print('✅ Random Forest loaded from Drive')

---
# MODEL 1 — RANDOM FOREST

A bagging ensemble of decision trees offering robustness against overfitting  
and interpretable feature importance scoring (Géron, 2020).  
Trained on full 78 features (SMOTE-balanced).

> ⏱ **Expected training time:** 15–30 minutes on T4 GPU (CPU-based, uses all cores)

## CELL 4 — Train Random Forest

In [ ]:
print('Training Random Forest...')
print(f'Training set: {X_train_sm.shape[0]:,} rows × {X_train_sm.shape[1]} features')
print('This may take 15-30 minutes. Do not close the tab.\n')

t0 = time.time()

rf = RandomForestClassifier(
    n_estimators=200,        # 200 trees — good balance of accuracy vs time
    max_depth=None,          # Full depth — let trees grow freely
    min_samples_leaf=2,      # Slight regularisation to prevent overfitting
    max_features='sqrt',     # Standard for classification
    class_weight='balanced', # Extra guard against class imbalance
    random_state=42,
    n_jobs=-1,               # Use all CPU cores
    verbose=1
)

rf.fit(X_train_sm, y_train_sm)
rf_time = time.time() - t0

print(f'\n✅ Random Forest trained in {rf_time:.1f}s ({rf_time/60:.1f} min)')
joblib.dump(rf, f'{MODEL_DIR}random_forest.pkl')
print('✅ Model saved to Drive')

Training Random Forest...
Training set: 5,836,260 rows × 78 features
This may take 15-30 minutes. Do not close the tab.



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed: 15.7min


## CELL 5 — Evaluate Random Forest

In [ ]:
y_pred_rf   = rf.predict(X_test)
y_prob_rf   = rf.predict_proba(X_test)

rf_results = evaluate_model('Random Forest', y_test, y_pred_rf, y_prob_rf, rf_time)
all_results.append(rf_results)

plot_confusion_matrix('Random Forest', y_test, y_pred_rf, 'cm_random_forest.png')
plot_pr_curves('Random Forest', y_test, y_prob_rf, 'pr_random_forest.png')

## CELL 6 — Random Forest Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)
top20 = importances.nlargest(20).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top20.index, top20.values, color=TEAL, edgecolor='white')
ax.set_xlabel('Feature Importance (Gini Impurity Reduction)', fontsize=12)
ax.set_title('Random Forest — Top 20 Feature Importances', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top20.values):
    ax.text(bar.get_width() + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved')

---
# MODEL 2 — XGBOOST

A sequential gradient boosting algorithm with state-of-the-art performance  
on structured tabular data and native SHAP compatibility (Kumar & Singh, 2023).  
Trained on full 78 features (SMOTE-balanced).

> ⏱ **Expected training time:** 20–40 minutes on T4 GPU

## CELL 7 — Train XGBoost

In [ ]:
print('Training XGBoost...')
print(f'Training set: {X_train_sm.shape[0]:,} rows × {X_train_sm.shape[1]} features')
print('This may take 20-40 minutes. Do not close the tab.\n')

t0 = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,          # 300 boosting rounds
    max_depth=6,               # Standard depth — prevents overfitting
    learning_rate=0.1,         # Shrinkage rate
    subsample=0.8,             # Row subsampling per tree
    colsample_bytree=0.8,      # Feature subsampling per tree
    min_child_weight=3,        # Min samples in leaf
    gamma=0.1,                 # Minimum loss reduction for split
    reg_alpha=0.1,             # L1 regularisation
    reg_lambda=1.0,            # L2 regularisation
    objective='multi:softprob',
    num_class=N_CLASSES,
    eval_metric='mlogloss',
    use_label_encoder=False,
    tree_method='hist',        # Faster histogram-based method
    device='cuda',             # Use GPU if available, falls back to CPU
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

xgb_model.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_test, y_test)],
    verbose=50  # Print every 50 rounds
)
xgb_time = time.time() - t0

print(f'\n✅ XGBoost trained in {xgb_time:.1f}s ({xgb_time/60:.1f} min)')
xgb_model.save_model(f'{MODEL_DIR}xgboost_model.json')
print('✅ Model saved to Drive')

## CELL 8 — Evaluate XGBoost

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)

xgb_results = evaluate_model('XGBoost', y_test, y_pred_xgb, y_prob_xgb, xgb_time)
all_results.append(xgb_results)

plot_confusion_matrix('XGBoost', y_test, y_pred_xgb, 'cm_xgboost.png')
plot_pr_curves('XGBoost', y_test, y_prob_xgb, 'pr_xgboost.png')

## CELL 9 — XGBoost Learning Curve

In [ ]:
# Plot validation loss across boosting rounds
results_dict = xgb_model.evals_result()
val_loss = results_dict['validation_0']['mlogloss']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(val_loss, color=TEAL, linewidth=2, label='Validation Log Loss')
ax.set_xlabel('Boosting Round', fontsize=12)
ax.set_ylabel('Multiclass Log Loss', fontsize=12)
ax.set_title('XGBoost — Validation Loss Across Boosting Rounds', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}xgb_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
# MODEL 3 — LSTM

Long Short-Term Memory recurrent neural network for detecting temporally  
distributed attack patterns (Yin et al., 2017). Trained on PCA-reduced  
16-component features reshaped as time sequences.

> ⏱ **Expected training time:** 20–40 minutes on T4 GPU  
> ⚠️ LSTM requires GPU — ensure T4 runtime is selected (Runtime → Change runtime type)

## CELL 10 — Prepare LSTM Inputs

LSTM expects 3D input: (samples, timesteps, features).  
We treat each network flow's 16 PCA components as a sequence of 16 timesteps × 1 feature.

In [ ]:
# Reshape PCA features for LSTM: (samples, timesteps=16, features=1)
X_train_lstm = X_train_pca.reshape(X_train_pca.shape[0], X_train_pca.shape[1], 1)
X_test_lstm  = X_test_pca.reshape(X_test_pca.shape[0],  X_test_pca.shape[1],  1)

# One-hot encode labels for categorical crossentropy
y_train_cat = to_categorical(y_train_sm, num_classes=N_CLASSES)
y_test_cat  = to_categorical(y_test,     num_classes=N_CLASSES)

print(f'LSTM input shapes:')
print(f'  X_train_lstm: {X_train_lstm.shape}  (samples, timesteps, features)')
print(f'  X_test_lstm:  {X_test_lstm.shape}')
print(f'  y_train_cat:  {y_train_cat.shape}   (one-hot encoded)')
print(f'  Timesteps: {X_train_lstm.shape[1]} | Features per step: {X_train_lstm.shape[2]}')

## CELL 11 — Build and Train LSTM

In [ ]:
# ── Model architecture ─────────────────────────────────────────────────────
lstm_model = Sequential([
    # Layer 1: LSTM with 128 units, return sequences for stacking
    LSTM(128, input_shape=(X_train_lstm.shape[1], 1),
         return_sequences=True, name='lstm_1'),
    BatchNormalization(name='bn_1'),
    Dropout(0.3, name='dropout_1'),

    # Layer 2: LSTM with 64 units
    LSTM(64, return_sequences=False, name='lstm_2'),
    BatchNormalization(name='bn_2'),
    Dropout(0.3, name='dropout_2'),

    # Dense layer
    Dense(64, activation='relu', name='dense_1'),
    Dropout(0.2, name='dropout_3'),

    # Output layer: 10 classes with softmax
    Dense(N_CLASSES, activation='softmax', name='output')
], name='LSTM_CyberRisk')

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

# ── Callbacks ──────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=3, min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        filepath=f'{MODEL_DIR}lstm_best.keras',
        monitor='val_loss', save_best_only=True, verbose=1
    )
]

# ── Training ───────────────────────────────────────────────────────────────
print('\nTraining LSTM...')
print('Early stopping enabled (patience=5 epochs)\n')
t0 = time.time()

history = lstm_model.fit(
    X_train_lstm, y_train_cat,
    validation_data=(X_test_lstm, y_test_cat),
    epochs=30,
    batch_size=1024,   # Large batch for speed on GPU
    callbacks=callbacks,
    verbose=1
)

lstm_time = time.time() - t0
print(f'\n✅ LSTM trained in {lstm_time:.1f}s ({lstm_time/60:.1f} min)')
lstm_model.save(f'{MODEL_DIR}lstm_model.keras')
print('✅ Model saved to Drive')

## CELL 12 — LSTM Training History Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'],     color=TEAL,      linewidth=2, label='Training Loss')
axes[0].plot(history.history['val_loss'], color=DARK_TEAL, linewidth=2, linestyle='--', label='Validation Loss')
axes[0].set_title('LSTM — Loss per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Categorical Crossentropy')
axes[0].legend()

# Accuracy
axes[1].plot(history.history['accuracy'],     color=TEAL,      linewidth=2, label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], color=DARK_TEAL, linewidth=2, linestyle='--', label='Validation Accuracy')
axes[1].set_title('LSTM — Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.suptitle('LSTM Training History — CyberRisk Framework', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}lstm_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 13 — Evaluate LSTM

In [ ]:
y_prob_lstm = lstm_model.predict(X_test_lstm, batch_size=2048, verbose=1)
y_pred_lstm = np.argmax(y_prob_lstm, axis=1)

lstm_results = evaluate_model('LSTM', y_test, y_pred_lstm, y_prob_lstm, lstm_time)
all_results.append(lstm_results)

plot_confusion_matrix('LSTM', y_test, y_pred_lstm, 'cm_lstm.png')
plot_pr_curves('LSTM', y_test, y_prob_lstm, 'pr_lstm.png')

---
# MODEL 4 — ISOLATION FOREST

Unsupervised anomaly detector for identifying unusual network behaviour  
without requiring labelled training data. Complements supervised models  
for zero-day and novel attack detection.

> ⏱ **Expected training time:** 5–10 minutes  
> Note: Isolation Forest produces binary outputs (normal / anomaly) not multi-class labels.

## CELL 14 — Train Isolation Forest

In [ ]:
# Isolation Forest trains on BENIGN traffic only (unsupervised: learns what "normal" looks like)
# Then flags deviations as anomalies
benign_mask = y_train_sm == 0  # Class 0 = BENIGN
X_train_benign = X_train_sm[benign_mask]

print(f'Training Isolation Forest on BENIGN traffic only...')
print(f'Training samples (BENIGN): {X_train_benign.shape[0]:,}')

t0 = time.time()

iso_forest = IsolationForest(
    n_estimators=200,
    contamination=0.267,  # ~26.7% of full dataset is attack traffic
    max_samples='auto',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

iso_forest.fit(X_train_benign)
iso_time = time.time() - t0

print(f'\n✅ Isolation Forest trained in {iso_time:.1f}s')
joblib.dump(iso_forest, f'{MODEL_DIR}isolation_forest.pkl')
print('✅ Model saved to Drive')

## CELL 15 — Evaluate Isolation Forest

In [ ]:
# Isolation Forest: +1 = normal (BENIGN), -1 = anomaly (ATTACK)
iso_raw_pred = iso_forest.predict(X_test)
iso_scores   = iso_forest.decision_function(X_test)  # Anomaly score

# Convert to binary: 0 = BENIGN, 1 = ATTACK (any)
y_pred_iso_binary = np.where(iso_raw_pred == 1, 0, 1)  # 1→BENIGN(0), -1→ATTACK(1)
y_test_binary     = np.where(y_test == 0, 0, 1)         # True: 0=BENIGN, 1=any attack

from sklearn.metrics import classification_report, confusion_matrix

print('=== ISOLATION FOREST — Binary Evaluation (BENIGN vs ATTACK) ===')
print(classification_report(y_test_binary, y_pred_iso_binary,
                             target_names=['BENIGN', 'ATTACK (any)']))

# Binary F1
iso_f1  = f1_score(y_test_binary, y_pred_iso_binary, average='macro')
iso_fpr_val = confusion_matrix(y_test_binary, y_pred_iso_binary)
fp = iso_fpr_val[0, 1]
tn = iso_fpr_val[0, 0]
iso_fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

iso_results = {
    'model': 'Isolation Forest',
    'macro_f1': iso_f1,
    'weighted_f1': f1_score(y_test_binary, y_pred_iso_binary, average='weighted'),
    'macro_precision': precision_score(y_test_binary, y_pred_iso_binary, average='macro', zero_division=0),
    'macro_recall': recall_score(y_test_binary, y_pred_iso_binary, average='macro', zero_division=0),
    'fpr': iso_fpr,
    'auc_roc': roc_auc_score(y_test_binary, -iso_scores),  # Negative: lower score = more anomalous
    'training_time_s': iso_time
}
all_results.append(iso_results)

print(f'Macro F1:   {iso_f1:.4f}')
print(f'FPR:        {iso_fpr:.4f}')
print(f'AUC-ROC:    {iso_results["auc_roc"]:.4f}')
print(f'Training time: {iso_time:.1f}s')

# Confusion matrix
cm_iso = confusion_matrix(y_test_binary, y_pred_iso_binary)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_iso, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['BENIGN', 'ATTACK'], yticklabels=['BENIGN', 'ATTACK'], ax=ax)
ax.set_title('Isolation Forest — Confusion Matrix (Binary)', fontsize=13, fontweight='bold')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}cm_isolation_forest.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 16 — Isolation Forest Anomaly Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

benign_scores = iso_scores[y_test == 0]
attack_scores = iso_scores[y_test != 0]

ax.hist(benign_scores, bins=100, alpha=0.6, color=TEAL,      label='BENIGN',       density=True)
ax.hist(attack_scores, bins=100, alpha=0.6, color=DARK_TEAL, label='ATTACK (any)', density=True)
ax.axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Decision boundary (0)')
ax.set_xlabel('Anomaly Score (higher = more normal)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Isolation Forest — Anomaly Score Distribution\nBENIGN vs ATTACK Traffic',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}iso_anomaly_scores.png', dpi=150, bbox_inches='tight')
plt.show()

---
# SHAP EXPLAINABILITY ANALYSIS

SHAP (SHapley Additive exPlanations) provides feature-level attribution  
for individual model predictions, satisfying DORA Article 13 auditability  
requirements (Arrieta et al., 2020).

## CELL 17 — SHAP for Random Forest

In [ ]:
print('Computing SHAP values for Random Forest...')
print('Using 200-record sample for speed\n')

# Reduced sample size — 200 is sufficient for publication-quality SHAP plots
shap_sample_idx = np.random.choice(X_test.shape[0], size=200, replace=False)
X_shap = X_test[shap_sample_idx]
y_shap = y_test[shap_sample_idx]

# Use interventional path_dependency for speed
rf_explainer = shap.TreeExplainer(rf, feature_perturbation='tree_path_dependent')
rf_shap_vals = rf_explainer.shap_values(X_shap, check_additivity=False)

print(f'SHAP values computed — shape per class: {rf_shap_vals[0].shape}')

# Plot 1: DDoS class summary
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    rf_shap_vals[1], X_shap,
    feature_names=feature_names,
    max_display=20, show=False
)
plt.title('SHAP Summary — Random Forest (DDoS Class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}shap_rf_ddos.png', dpi=150, bbox_inches='tight')
plt.show()

# Plot 2: Global importance
mean_shap = np.mean([np.abs(rf_shap_vals[i]).mean(axis=0) for i in range(N_CLASSES)], axis=0)
shap_importance = pd.Series(mean_shap, index=feature_names).sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 8))
shap_importance.plot(kind='barh', ax=ax, color=TEAL, edgecolor='white')
ax.set_title('SHAP Feature Importance — Random Forest\nMean |SHAP| across all 10 classes',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}shap_rf_global.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP Random Forest plots saved')

## CELL 18 — SHAP for XGBoost

In [ ]:
print('Computing SHAP values for XGBoost...')

xgb_explainer = shap.TreeExplainer(xgb_model)
xgb_shap_vals = xgb_explainer.shap_values(X_shap)

# Summary plot — all classes combined
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    xgb_shap_vals[1],   # DDoS class for direct comparison with RF
    X_shap,
    feature_names=feature_names,
    max_display=20,
    show=False
)
plt.title('SHAP Summary — XGBoost (DDoS Class)\nFeature impact on DDoS prediction',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}shap_xgb_ddos.png', dpi=150, bbox_inches='tight')
plt.show()

# Global importance comparison: RF vs XGBoost
mean_shap_xgb = np.mean([np.abs(xgb_shap_vals[i]).mean(axis=0) for i in range(N_CLASSES)], axis=0)
xgb_shap_imp = pd.Series(mean_shap_xgb, index=feature_names)

# Top 15 features in both models
top15 = shap_importance.index[-15:].tolist()
compare_df = pd.DataFrame({
    'Random Forest': pd.Series(mean_shap, index=feature_names)[top15],
    'XGBoost': xgb_shap_imp[top15]
})

fig, ax = plt.subplots(figsize=(12, 7))
compare_df.plot(kind='barh', ax=ax, color=[TEAL, DARK_TEAL], edgecolor='white')
ax.set_title('SHAP Feature Importance — Random Forest vs XGBoost\nMean |SHAP| (top 15 features, all classes)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}shap_rf_vs_xgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ SHAP XGBoost plots saved')

---
# COMPARATIVE MODEL EVALUATION

Cross-model performance comparison for dissertation Chapter 4.

## CELL 19 — Comparative Metrics Table

In [ ]:
# Build comparison dataframe
comp_df = pd.DataFrame(all_results).set_index('model')
comp_df.columns = ['Macro F1', 'Weighted F1', 'Macro Precision', 'Macro Recall',
                   'Mean FPR', 'AUC-ROC', 'Training Time (s)']

# Round for display
display_df = comp_df.round(4)

print('='*75)
print('  COMPARATIVE MODEL EVALUATION — All Four Architectures')
print('='*75)
print(display_df.to_string())

# Save to CSV
comp_df.to_csv(f'{MODEL_DIR}model_comparison.csv')
print('\n✅ Comparison table saved to Drive')

# Highlight best per metric
print('\nBest model per metric:')
for col in ['Macro F1', 'Weighted F1', 'AUC-ROC']:
    if col in comp_df.columns:
        best = comp_df[col].idxmax()
        val  = comp_df[col].max()
        print(f'  {col}: {best} ({val:.4f})')
print(f'  Lowest FPR: {comp_df["Mean FPR"].idxmin()} ({comp_df["Mean FPR"].min():.4f})')
print(f'  Fastest: {comp_df["Training Time (s)"].idxmin()} ({comp_df["Training Time (s)"].min():.1f}s)')

## CELL 20 — Comparative Visualisation

In [ ]:
metrics = ['Macro F1', 'Weighted F1', 'Macro Precision', 'Macro Recall', 'AUC-ROC']
models  = comp_df.index.tolist()
colors  = [TEAL, DARK_TEAL, '#05A8A8', '#037775']

fig, axes = plt.subplots(1, len(metrics), figsize=(20, 6))

for ax, metric in zip(axes, metrics):
    if metric in comp_df.columns:
        vals = comp_df[metric].values
        bars = ax.bar(models, vals, color=colors, edgecolor='white', width=0.6)
        ax.set_title(metric, fontsize=11, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.set_xticklabels(models, rotation=30, ha='right', fontsize=9)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.suptitle('Comparative Model Evaluation — Cyber Risk Analytics Framework\nCICIDS2017 Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Comparison chart saved')

## CELL 21 — F1 Per Class Heatmap (RF vs XGBoost vs LSTM)

In [ ]:
from sklearn.metrics import f1_score

f1_per_class = {}
for name, y_pred in [
    ('Random Forest', y_pred_rf),
    ('XGBoost',       y_pred_xgb),
    ('LSTM',          y_pred_lstm)
]:
    f1_per_class[name] = f1_score(y_test, y_pred, average=None, zero_division=0)

f1_df = pd.DataFrame(f1_per_class, index=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    f1_df, annot=True, fmt='.3f',
    cmap='YlGn', vmin=0, vmax=1,
    linewidths=0.5, ax=ax, cbar=True
)
ax.set_title('Per-Class F1-Score — Random Forest vs XGBoost vs LSTM\n(CICIDS2017 Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Attack Category', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}f1_per_class_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Per-class F1 heatmap saved')

---
# RISK TIER DECISION-SUPPORT FRAMEWORK

Translates ML model outputs into governance-compatible risk tiers  
aligned with ISO 31000 and DORA Article 13 (O5 in the research objectives).

## CELL 22 — Define Risk Tier Mapping

In [ ]:
# Risk tier mapping: attack category → risk score (0–100) → tier
# Based on CVSS-style severity, DORA Article 13 threat impact classification,
# and banking operational risk taxonomy

RISK_SCORE_MAP = {
    'BENIGN':                   0,
    'Web Attack - SQL Injection': 95,  # Critical — direct data exfiltration risk
    'DDoS':                      90,  # Critical — service availability impact
    'Web Attack - Brute Force':  80,  # High     — credential compromise
    'Web Attack - XSS':          75,  # High     — client-side code injection
    'SSH-Patator':               70,  # High     — infrastructure credential attack
    'FTP-Patator':               65,  # High     — service credential attack
    'DoS Hulk':                  60,  # Medium   — volumetric DoS
    'DoS Slowhttptest':          55,  # Medium   — slow DoS variant
    'DoS Slowloris':             50,  # Medium   — slow DoS variant
}

TIER_CONFIG = {
    'Critical': {'range': (85, 100), 'color': '#D32F2F', 'response': 'Immediate escalation. Isolate affected systems. Invoke Incident Response Plan. Notify CISO and regulators per DORA Article 19.'},
    'High':     {'range': (65, 84),  'color': '#F57C00', 'response': 'Alert SOC within 15 minutes. Block source IP. Initiate forensic logging. Review affected accounts.'},
    'Medium':   {'range': (40, 64),  'color': '#FBC02D', 'response': 'Log and monitor. Escalate if pattern persists beyond 30 minutes. Review firewall rules.'},
    'Low':      {'range': (0, 39),   'color': '#388E3C', 'response': 'Record in risk register. No immediate action required. Review in next daily report.'},
}

def get_risk_tier(score):
    for tier, cfg in TIER_CONFIG.items():
        lo, hi = cfg['range']
        if lo <= score <= hi:
            return tier
    return 'Low'

def classify_event(predicted_label, confidence_score):
    """Map a model prediction to a risk tier decision."""
    base_score = RISK_SCORE_MAP.get(predicted_label, 0)
    # Adjust score by model confidence (high confidence → full score)
    adjusted_score = base_score * confidence_score
    tier = get_risk_tier(adjusted_score)
    return {
        'predicted_class':  predicted_label,
        'confidence':       round(confidence_score, 4),
        'base_risk_score':  base_score,
        'adjusted_score':   round(adjusted_score, 1),
        'risk_tier':        tier,
        'response':         TIER_CONFIG[tier]['response']
    }

print('✅ Risk tier framework defined')
print('\nRisk tier ranges:')
for tier, cfg in TIER_CONFIG.items():
    print(f'  {tier}: score {cfg["range"][0]}–{cfg["range"][1]}')

## CELL 23 — Apply Risk Framework to Test Set (Best Model)

In [ ]:
# Apply using best performing supervised model (RF or XGBoost — check Cell 19)
# Defaulting to XGBoost here; change to y_pred_rf / y_prob_rf if RF was better

print('Applying Risk Tier Framework to test set predictions (XGBoost)...')

risk_records = []
for i in range(len(y_test)):
    pred_class = CLASS_NAMES[y_pred_xgb[i]]
    confidence = y_prob_xgb[i].max()  # Highest class probability
    record = classify_event(pred_class, confidence)
    record['true_class'] = CLASS_NAMES[y_test[i]]
    risk_records.append(record)

risk_df = pd.DataFrame(risk_records)

print(f'\n✅ Risk classification complete for {len(risk_df):,} network flow events')
print('\nRisk tier distribution:')
tier_counts = risk_df['risk_tier'].value_counts()
for tier in ['Critical', 'High', 'Medium', 'Low']:
    count = tier_counts.get(tier, 0)
    pct = count / len(risk_df) * 100
    print(f'  {tier:10s}: {count:8,} events ({pct:.2f}%)')

# Save
risk_df.to_csv(f'{MODEL_DIR}risk_tier_classifications.csv', index=False)
print('\n✅ Risk classifications saved to Drive')

## CELL 24 — Risk Tier Visualisation

In [ ]:
tier_order  = ['Critical', 'High', 'Medium', 'Low']
tier_colors = [TIER_CONFIG[t]['color'] for t in tier_order]
tier_vals   = [tier_counts.get(t, 0) for t in tier_order]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
bars = axes[0].bar(tier_order, tier_vals, color=tier_colors, edgecolor='white', width=0.6)
for bar, v in zip(bars, tier_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                f'{v:,}\n({v/len(risk_df)*100:.1f}%)',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Risk Tier Distribution\n(XGBoost predictions, test set)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Network Flow Events', fontsize=11)
axes[0].set_xlabel('Risk Tier', fontsize=11)

# Pie chart
axes[1].pie(
    tier_vals,
    labels=[f'{t}\n{v:,} ({v/len(risk_df)*100:.1f}%)' for t, v in zip(tier_order, tier_vals)],
    colors=tier_colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Risk Tier Proportions\n(ISO 31000 aligned)', fontsize=13, fontweight='bold')

plt.suptitle('Risk Tier Decision-Support Framework — DORA Article 13 Aligned\nCICIDS2017 Test Set | XGBoost Model',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}risk_tier_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## CELL 25 — Sample Risk Decision Report (10 events)

In [ ]:
# Sample 2 events from each tier for a governance-facing report
sample_events = []
for tier in ['Critical', 'High', 'Medium', 'Low']:
    tier_sample = risk_df[risk_df['risk_tier'] == tier].head(2)
    sample_events.append(tier_sample)

sample_df = pd.concat(sample_events, ignore_index=True)

print('='*100)
print('  SAMPLE RISK DECISION REPORT — Data-Driven Cyber Risk Analytics Framework')
print('  Institution: Digital Banking Environment | Framework: DORA Article 13 / ISO 31000')
print('='*100)

for _, row in sample_df.iterrows():
    tier_color_label = {'Critical': '🔴', 'High': '🟠', 'Medium': '🟡', 'Low': '🟢'}
    print(f"\n{tier_color_label[row['risk_tier']]} RISK TIER: {row['risk_tier'].upper()} | Score: {row['adjusted_score']} / 100")
    print(f"   Predicted Attack: {row['predicted_class']}  (True: {row['true_class']})")
    print(f"   Model Confidence: {row['confidence']*100:.1f}%")
    print(f"   Recommended Response: {row['response']}")
    print(f"   {'─'*90}")

# Also save formatted sample
sample_df.to_csv(f'{MODEL_DIR}sample_risk_report.csv', index=False)
print('\n✅ Sample risk report saved to Drive')

---
## CELL 26 — Phase 2 Final Summary Report

In [ ]:
print('='*65)
print('  PHASE 2 COMPLETE — FINAL SUMMARY')
print('='*65)

print(f'''
MODELS TRAINED AND EVALUATED:
  ✅ Random Forest    (n_estimators=200, 78 features, SMOTE-balanced)
  ✅ XGBoost          (n_estimators=300, 78 features, SMOTE-balanced)
  ✅ LSTM             (2-layer, 16 PCA components, GPU-accelerated)
  ✅ Isolation Forest (unsupervised, contamination=0.267)

EVALUATION METRICS COMPUTED:
  ✅ Macro & Weighted F1-score
  ✅ Precision, Recall (macro)
  ✅ AUC-ROC (macro, OvR)
  ✅ False Positive Rate
  ✅ Per-class classification reports
  ✅ Confusion matrices (counts + normalised)
  ✅ Precision-Recall curves

EXPLAINABILITY:
  ✅ SHAP TreeExplainer — Random Forest (global + DDoS class)
  ✅ SHAP TreeExplainer — XGBoost (global + RF comparison)

RISK FRAMEWORK:
  ✅ 4-tier risk classification (Critical/High/Medium/Low)
  ✅ ISO 31000 aligned response recommendations
  ✅ DORA Article 13 compliant audit trail
  ✅ {len(risk_df):,} network events classified
''')

print('COMPARATIVE RESULTS:')
print(comp_df[['Macro F1', 'AUC-ROC', 'Mean FPR']].round(4).to_string())

print(f'''
FILES SAVED TO DRIVE ({MODEL_DIR}):
  random_forest.pkl            — Trained RF model
  xgboost_model.json           — Trained XGBoost model
  lstm_model.keras             — Trained LSTM model
  isolation_forest.pkl         — Trained Isolation Forest
  model_comparison.csv         — Metrics comparison table
  risk_tier_classifications.csv — Full test set risk decisions
  sample_risk_report.csv        — Sample governance report

FIGURES SAVED TO DRIVE ({FIG_DIR}):
  cm_*.png                     — Confusion matrices (4 models)
  pr_*.png                     — Precision-Recall curves
  shap_*.png                   — SHAP explainability plots
  model_comparison.png         — Cross-model metrics chart
  f1_per_class_heatmap.png     — Per-class F1 comparison
  risk_tier_distribution.png   — Risk tier output chart

NEXT: Phase 3 — Dissertation write-up (Chapter 4: Results & Chapter 5: Discussion)
''')
print('='*65)